# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql ( a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.  
- Lab due date is TBD ( due dates will be updated in the readme for the class repo )

In [0]:
# Define the name of the new catalog
catalog = 'taxi_eda_db'

# define variables for the trips data
schema = 'yellow_taxi_trips'
volume = 'data'
file_name = 'yellow_tripdata_2019-01.parquet'
table_name = 'tbl_yellow_taxi_trips'
path_volume = '/Volumes/' + catalog + "/" + schema + '/' + volume
path_table =  catalog + "." + schema
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

In [0]:

# create the catalog/schema/volume
spark.sql('create catalog if not exists ' + catalog)
spark.sql('create schema if not exists ' + catalog + '.' + schema)
spark.sql('create volume if not exists ' + catalog + '.' + schema + '.' + volume)

DataFrame[]

In [0]:
# Get the data
dbutils.fs.cp(f"{download_url}", f"{path_volume}" + "/" + f"{file_name}")

True

In [0]:

# create the dataframe
df_trips = spark.read.parquet(f"{path_volume}/{file_name}",
  header=True,
  inferSchema=True,
  sep=",")

In [0]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [0]:
# Add a column that creates a unique key to identify each record in order to answer questions about individual trips
from pyspark.sql.functions import monotonically_increasing_id
# Add key column
df_trips = df_trips.withColumn("KEY", monotonically_increasing_id())

In [0]:
# Which trip has the highest passanger count
import pyspark.sql.functions as sf
# highest passanger count
df_trips.select(sf.max("passenger_count")).show()

+--------------------+
|max(passenger_count)|
+--------------------+
|                 9.0|
+--------------------+



In [0]:
# What is the Average passanger count
# Average passanger count
df_trips.select(sf.avg("passenger_count")).show()

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [0]:
# Shortest/longest trip by distance? by time?
print("Shortest/ Longest trip by distance:")
df_trips.select(sf.min("trip_distance")).show()
df_trips.select(sf.max("trip_distance")).show()

print("\nShortest/ Longest trip by time:")

df_trips = df_trips.withColumn("Diff_tpep_datetime", sf.col("tpep_dropoff_datetime") - sf.col("tpep_pickup_datetime")) 

print("\nShortest/ Longest trip by time:")
df_trips.select(sf.min("Diff_tpep_datetime")).show()
df_trips.select(sf.max("Diff_tpep_datetime")).show()


Shortest/ Longest trip by distance:
+------------------+
|min(trip_distance)|
+------------------+
|               0.0|
+------------------+

+------------------+
|max(trip_distance)|
+------------------+
|             831.8|
+------------------+


Shortest/ Longest trip by time:

Shortest/ Longest trip by time:
+-----------------------+
|min(Diff_tpep_datetime)|
+-----------------------+
|   INTERVAL '-58 12:...|
+-----------------------+

+-----------------------+
|max(Diff_tpep_datetime)|
+-----------------------+
|   INTERVAL '30 07:2...|
+-----------------------+



In [0]:
# busiest day/slowest single day
print("Busiest day:")
df_trips_pickup = df_trips.withColumn("pickup_date", sf.to_date("tpep_pickup_datetime"))
df_trips_pickup.groupBy("pickup_date").count().orderBy(sf.desc("count")).limit(1).show()

print("Slowest day:")
df_trips_pickup.groupBy("pickup_date").count().orderBy(sf.asc("count")).limit(1).show()


Busiest day:
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
+-----------+------+

Slowest day:
+-----------+-----+
|pickup_date|count|
+-----------+-----+
| 2018-12-21|    1|
+-----------+-----+



In [0]:
# busiest/slowest time of day
print("Busiest time of day:")
df_trips_pickup = df_trips.withColumn("pickup_hour", sf.hour("tpep_pickup_datetime"))
df_trips_pickup.groupBy("pickup_hour").count().orderBy(sf.desc("count")).limit(1).show()

print("Slowest time of day:")
df_trips_pickup.groupBy("pickup_hour").count().orderBy(sf.asc("count")).limit(1).show()



Busiest time of day:
+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|515390|
+-----------+------+

Slowest time of day:
+-----------+-----+
|pickup_hour|count|
+-----------+-----+
|          4|61424|
+-----------+-----+



In [0]:
# busiest day/slowest day of week (on average)

day_week_avg = df_trips.withColumn("pickup_date", sf.dayofweek("tpep_pickup_datetime"))
day_week_avg = day_week_avg.withColumn("pickup_month", sf.month("tpep_pickup_datetime"))

df_counts = day_week_avg.groupBy("pickup_month", "pickup_date").count()
df_avg = (
    df_counts
    .groupBy("pickup_date")
    .agg(sf.avg("count").alias("avg_trips"))
    .orderBy("pickup_date")
)
df_avg.show()
print("Busiest day of week (on average):")
df_avg.orderBy(sf.desc("avg_trips")).limit(1).show()

print("Slowest day of week (on average):")
df_avg.orderBy(sf.asc("avg_trips")).limit(1).show()



+-----------+---------+
|pickup_date|avg_trips|
+-----------+---------+
|          1|214976.25|
|          2| 151353.5|
|          3| 201514.0|
|          4| 316316.0|
|          5| 678521.5|
|          6| 362405.0|
|          7| 504992.5|
+-----------+---------+

Busiest day of week (on average):
+-----------+---------+
|pickup_date|avg_trips|
+-----------+---------+
|          5| 678521.5|
+-----------+---------+

Slowest day of week (on average):
+-----------+---------+
|pickup_date|avg_trips|
+-----------+---------+
|          2| 151353.5|
+-----------+---------+



In [0]:
# Does trip distance or num passangers affect tip amount
df_corr = df_trips.select("trip_distance", "passenger_count", "tip_amount")

print("Correlation between trip_distance and tip_amount: ", df_corr.corr("trip_distance", "tip_amount"))
print("Correlation between passenger_count and tip_amount: ", df_corr.corr("passenger_count", "tip_amount"))

# Trip distance affect the tip amount.

Correlation between trip_distance and tip_amount:  0.5269200663652669
Correlation between passenger_count and tip_amount:  0.004431051585116288


In [0]:
# What was the highest "extra" charge and which trip
max_charge = df_trips.select(sf.max("extra"))
df_trips.filter(sf.col("extra") == max_charge.first()[0]).show()  

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount| extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    KEY|  Diff_tpep_datetime|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+--------------------+
|       1| 2019-01-23 08:58:09|  2019-01-23 08:58:09|            1.0|          0.0|       1.0|   

In [0]:
# Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?
# What is the average trip distance?
df_trips.select(sf.avg("trip_distance")).show()

+------------------+
|avg(trip_distance)|
+------------------+
|2.8301461681151574|
+------------------+



### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [0]:
# Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
catalog = 'taxi_eda_lookup'
schema  = 'yellow_taxi_trips_lookup'
volume  = 'data'

lookup_src_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv'
lookup_dst_name = 'taxi_zone_lookup.csv'
path_volume = f'/Volumes/{catalog}/{schema}/{volume}'
path_table  = f'{catalog}.{schema}'
lookup_dst_path = f'{path_volume}/{lookup_dst_name}'

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
spark.sql(f'CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}')
spark.sql(f'CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}')

dbutils.fs.cp(lookup_src_url, lookup_dst_path)

df_lookup = (spark.read
    .option('header', True)
    .option('inferSchema', True)
    .option('multiLine', False)
    .option('mode', 'PERMISSIVE')
    .csv(lookup_dst_path)
)

df_lookup.show()


+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [0]:
# which borough had most pickups? dropoffs?
df_join = df_trips.join(df_lookup, df_trips.PULocationID == df_lookup.LocationID, "left")
print("Borough with most pickups:")
df_join_pickups = df_join.groupBy("Borough", "tpep_pickup_datetime").count()
df_join_pickups.groupBy("Borough").agg(sf.sum("count").alias("total_pickups")).orderBy(sf.desc("total_pickups")).limit(1).show()
print("Borough with most dropoffs:")
df_join_dropoffs = df_join.groupBy("Borough", "tpep_dropoff_datetime").count()
df_join_dropoffs.groupBy("Borough").agg(sf.sum("count").alias("total_dropoffs")).orderBy(sf.desc("total_dropoffs")).limit(1).show()

Borough with most pickups:
+---------+-------------+
|  Borough|total_pickups|
+---------+-------------+
|Manhattan|      6950965|
+---------+-------------+

Borough with most dropoffs:
+---------+--------------+
|  Borough|total_dropoffs|
+---------+--------------+
|Manhattan|       6950965|
+---------+--------------+



In [0]:
# what are the busy/slow times by borough
df_busy = df_join.withColumn("pickup_date", sf.hour("tpep_pickup_datetime"))

print("Busy:")
df_filter = df_join.withColumn("pickup_hour", sf.hour("tpep_pickup_datetime"))
df_filter.groupBy("Borough", "pickup_hour").count().orderBy(sf.desc("count")).show()

print("Slow:")
df_filter.groupBy("Borough", "pickup_hour").count().orderBy(sf.asc("count")).show()

Busy:
+---------+-----------+------+
|  Borough|pickup_hour| count|
+---------+-----------+------+
|Manhattan|         18|471539|
|Manhattan|         19|432836|
|Manhattan|         17|426498|
|Manhattan|         15|409119|
|Manhattan|         14|391823|
|Manhattan|         20|382357|
|Manhattan|         16|377245|
|Manhattan|         21|367636|
|Manhattan|         12|367189|
|Manhattan|         13|366911|
|Manhattan|         11|346110|
|Manhattan|          8|338274|
|Manhattan|          9|333426|
|Manhattan|         22|329592|
|Manhattan|         10|327978|
|Manhattan|          7|272314|
|Manhattan|         23|248416|
|Manhattan|          0|183353|
|Manhattan|          6|155963|
|Manhattan|          1|134734|
+---------+-----------+------+
only showing top 20 rows
Slow:
+-------------+-----------+-----+
|      Borough|pickup_hour|count|
+-------------+-----------+-----+
|          EWR|          1|    1|
|          EWR|         23|    1|
|          EWR|          0|    2|
|Staten Island|

In [0]:
# what is the average trip distance by borough?
df_join.groupBy("Borough").agg(sf.avg("trip_distance")).orderBy(sf.desc("avg(trip_distance)")).show()

+-------------+------------------+
|      Borough|avg(trip_distance)|
+-------------+------------------+
|Staten Island|12.503601108033246|
|       Queens|11.283218499361993|
|        Bronx| 7.233194552098303|
|     Brooklyn| 4.787677275447492|
|          N/A| 3.193850899742941|
|          EWR| 2.641098654708519|
|      Unknown| 2.415464130400774|
|    Manhattan|2.2286693358402596|
+-------------+------------------+



In [0]:

# what is the average trip fare by borough?
df_join.groupBy("Borough").agg(sf.avg("total_amount")).orderBy(sf.desc("avg(total_amount)")).show()

+-------------+------------------+
|      Borough| avg(total_amount)|
+-------------+------------------+
|          EWR| 92.78352017937229|
|          N/A| 69.54899485860886|
|Staten Island| 53.58659279778373|
|       Queens| 44.45193604055996|
|        Bronx|29.306521979843616|
|     Brooklyn| 21.61839747567109|
|      Unknown| 18.16889928978967|
|    Manhattan|13.666141060205877|
+-------------+------------------+



In [0]:
#highest/lowest faire amounts for a trip, what burough is associated with the each
df_join.groupBy("Borough").agg(sf.max("total_amount")).orderBy(sf.desc("max(total_amount)")).limit(1).show()
df_join.groupBy("Borough").agg(sf.min("total_amount")).orderBy(sf.asc("min(total_amount)")).limit(1).show()

+---------+-----------------+
|  Borough|max(total_amount)|
+---------+-----------------+
|Manhattan|        623261.66|
+---------+-----------------+

+-------+-----------------+
|Borough|min(total_amount)|
+-------+-----------------+
| Queens|           -362.8|
+-------+-----------------+



In [0]:
# load the dataset from the most recently available january, is there a change to any of the average metrics.
catalog = 'taxi_eda_db'

# define variables for the trips data
schema = 'yellow_taxi_trips'
volume = 'data'
file_name = 'yellow_tripdata_2025-01.parquet'
table_name = 'tbl_yellow_taxi_trips'
path_volume = '/Volumes/' + catalog + "/" + schema + '/' + volume
path_table =  catalog + "." + schema
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'


# create the catalog/schema/volume
spark.sql('create catalog if not exists ' + catalog)
spark.sql('create schema if not exists ' + catalog + '.' + schema)
spark.sql('create volume if not exists ' + catalog + '.' + schema + '.' + volume)

# Get the data
dbutils.fs.cp(f"{download_url}", f"{path_volume}" + "/" + f"{file_name}")
# create the dataframe
df_trips_2025 = spark.read.parquet(f"{path_volume}/{file_name}",
  header=True,
  inferSchema=True,
  sep=",")

  

In [0]:
# which borough had most pickups? dropoffs?
df_join = df_trips_2025.join(df_lookup, df_trips_2025.PULocationID == df_lookup.LocationID, "left")
print("Borough with most pickups:")
df_join_pickups = df_join.groupBy("Borough", "tpep_pickup_datetime").count()
df_join_pickups.groupBy("Borough").agg(sf.sum("count").alias("total_pickups")).orderBy(sf.desc("total_pickups")).limit(1).show()
print("Borough with most dropoffs:")
df_join_dropoffs = df_join.groupBy("Borough", "tpep_dropoff_datetime").count()
df_join_dropoffs.groupBy("Borough").agg(sf.sum("count").alias("total_dropoffs")).orderBy(sf.desc("total_dropoffs")).limit(1).show()

Borough with most pickups:
+---------+-------------+
|  Borough|total_pickups|
+---------+-------------+
|Manhattan|      3089275|
+---------+-------------+

Borough with most dropoffs:
+---------+--------------+
|  Borough|total_dropoffs|
+---------+--------------+
|Manhattan|       3089275|
+---------+--------------+



In [0]:
# what are the busy/slow times by borough
df_busy = df_join.withColumn("pickup_date", sf.hour("tpep_pickup_datetime"))

print("Busy:")
df_filter = df_join.withColumn("pickup_hour", sf.hour("tpep_pickup_datetime"))
df_filter.groupBy("Borough", "pickup_hour").count().orderBy(sf.desc("count")).show()

print("Slow:")
df_filter.groupBy("Borough", "pickup_hour").count().orderBy(sf.asc("count")).show()


Busy:
+---------+-----------+------+
|  Borough|pickup_hour| count|
+---------+-----------+------+
|Manhattan|         18|246223|
|Manhattan|         17|230505|
|Manhattan|         19|199780|
|Manhattan|         16|192587|
|Manhattan|         15|189469|
|Manhattan|         21|184486|
|Manhattan|         14|179492|
|Manhattan|         20|173826|
|Manhattan|         13|167715|
|Manhattan|         22|161166|
|Manhattan|         12|158489|
|Manhattan|         11|145565|
|Manhattan|         10|132653|
|Manhattan|          9|127998|
|Manhattan|          8|124469|
|Manhattan|         23|115987|
|Manhattan|          7| 85717|
|Manhattan|          0| 80066|
|Manhattan|          1| 57720|
|Manhattan|          2| 39731|
+---------+-----------+------+
only showing top 20 rows
Slow:
+-------------+-----------+-----+
|      Borough|pickup_hour|count|
+-------------+-----------+-----+
|          EWR|          1|    1|
|          EWR|          2|    1|
|Staten Island|         17|    2|
|          EWR|

In [0]:
# what is the average trip distance by borough?
df_join.groupBy("Borough").agg(sf.avg("trip_distance")).orderBy(sf.desc("avg(trip_distance)")).show()
# what is the average trip fare by borough?
df_join.groupBy("Borough").agg(sf.avg("total_amount")).orderBy(sf.desc("avg(total_amount)")).show()

+-------------+------------------+
|      Borough|avg(trip_distance)|
+-------------+------------------+
|        Bronx| 65.91340478936304|
|          N/A|28.235833333333336|
|     Brooklyn|24.807041925231157|
|       Queens|13.360874380478473|
|Staten Island| 8.305703124999999|
|    Manhattan|4.4439188903536015|
|      Unknown| 3.202056258444915|
|          EWR|0.8962864721485414|
+-------------+------------------+

+-------------+------------------+
|      Borough| avg(total_amount)|
+-------------+------------------+
|          EWR| 94.33787798408491|
|          N/A| 89.56308695652153|
|       Queens| 65.31643630544527|
|Staten Island| 36.24363281249999|
|        Bronx|31.342387219320436|
|      Unknown|27.671546493059907|
|     Brooklyn|27.117854850915382|
|    Manhattan| 21.71712872761903|
+-------------+------------------+



In [0]:
#highest/lowest faire amounts for a trip, what burough is associated with the each
df_join.groupBy("Borough").agg(sf.max("total_amount")).orderBy(sf.desc("max(total_amount)")).limit(1).show()
df_join.groupBy("Borough").agg(sf.min("total_amount")).orderBy(sf.asc("min(total_amount)")).limit(1).show()

+-------+-----------------+
|Borough|max(total_amount)|
+-------+-----------------+
| Queens|        863380.37|
+-------+-----------------+

+-------+-----------------+
|Borough|min(total_amount)|
+-------+-----------------+
| Queens|           -901.0|
+-------+-----------------+



### Comparison between January 2025 and January 2019:

- We see that we have more collections and deposits in 2025 than in 2019 (but this is for the same district).
- We still have the same busiest and slowest period between 2019 and 2025.
- The average distance of a trip has exploded, going from 5.78 in 2019 to 18.6 in 2025. Consequently, the average price of the tip also increases, going from 42.88 in 2019 to 49.15 in 2025.
- The highest fair amount is higher in January 2025 than in January 2019. The lowest fair amount is lower in January 2025 than in January 2019.



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [0]:
df_trips.createOrReplaceTempView("trips")
df_lookup.createOrReplaceTempView("lookup")
# which borough had most pickups? dropoffs?
most_pickups = spark.sql("""SELECT Borough, count(*) AS total_pickups FROM trips JOIN lookup ON trips.PULocationID = lookup.LocationID GROUP BY Borough ORDER BY total_pickups DESC LIMIT 1 """)

most_dropoffs = spark.sql("""SELECT Borough, count(*) AS total_dropoffs FROM trips t JOIN lookup l ON t.DOLocationID = l.LocationID GROUP BY Borough ORDER BY total_dropoffs DESC LIMIT 1 """)

most_pickups.show()
most_dropoffs.show()


+---------+-------------+
|  Borough|total_pickups|
+---------+-------------+
|Manhattan|      6950965|
+---------+-------------+

+---------+--------------+
|  Borough|total_dropoffs|
+---------+--------------+
|Manhattan|       6817355|
+---------+--------------+



In [0]:
# highest passanger count
result = spark.sql("""SELECT MAX(passenger_count) FROM trips""")
result.show() 


+--------------------+
|MAX(passenger_count)|
+--------------------+
|                 9.0|
+--------------------+



In [0]:
# What is the Average passanger count
result = spark.sql("""SELECT AVG(passenger_count) FROM trips""")
result.show()

+--------------------+
|AVG(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+




### Part 4

As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.


In [0]:
# what is the average trip distance by borough?
avg_distance = df_join.groupBy("Borough").agg(sf.avg("trip_distance")).orderBy(sf.desc("avg(trip_distance)"))

avg_distance.plot(kind="bar", x="Borough", y="avg(trip_distance)", title="average trip distance by borough")

In [0]:
# what is the average trip fare by borough?
result = df_join.groupBy("Borough").agg(sf.avg("total_amount")).orderBy(sf.desc("avg(total_amount)"))

result.plot(kind="pie", x="Borough", y="avg(total_amount)", title="average trip fare by borough")

In [0]:
# busiest day/slowest day of week (on average)

day_week_avg = df_trips.withColumn("pickup_date", sf.dayofweek("tpep_pickup_datetime"))
day_week_avg = day_week_avg.withColumn("pickup_month", sf.month("tpep_pickup_datetime"))

df_counts = day_week_avg.groupBy("pickup_month", "pickup_date").count()
df_avg = (
    df_counts
    .groupBy("pickup_date")
    .agg(sf.avg("count").alias("avg_trips"))
    .orderBy("pickup_date")
)
df_avg.plot(kind="bar", x="pickup_date", y="avg_trips", title="average trips by day of week")



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- Explore a dataset/datasets of your choosing